In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# !pip install -q kagglehub tensorflow
import tensorflow as tf
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from tqdm import tqdm
from tensorflow.keras.utils import load_img, img_to_array
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("done")

/Users/ozgeyavuz/490/cat_or_dog_model_dataset/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


done


/Users/ozgeyavuz/490/cat_or_dog_model_dataset/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Download the model
model_path = "/Users/ozgeyavuz/490/cat_or_dog_model_dataset/cats-vs-dogs-classifier-tensorflow2-fine-tuned-mobilenetv2-v1/cats_dogs_finetuned_FT.keras"


# Load the model
model = tf.keras.models.load_model(model_path)
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,314,629 (20.27 MB)

 Trainable params: 1,527,681 (5.83 MB)

 Non-trainable params: 731,584 (2.79 MB)

 Optimizer params: 3,055,364 (11.66 MB)

In [4]:
for layer in model.layers:
    print(layer.name)



input_layer_2
mobilenetv2_1.00_224
global_average_pooling2d
dropout
dense


In [5]:
from tensorflow.keras.models import Model

# 1) Image load + preprocess
def load_and_preprocess(img_path):
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
    img = tf.keras.preprocessing.image.img_to_array(img)
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    return np.expand_dims(img, axis=0)

# 2) Feature vector model (GAP output)
def get_feature_model(model):
    gap_layer = model.get_layer('global_average_pooling2d')
    return Model(inputs=model.input, outputs=gap_layer.output)

# # ---- Example Usage ----
# feature_model = get_feature_model(model)
# features = feature_model.predict(load_and_preprocess("PetImages/test/cats/cat_56.jpg"))
# print(features)   # (1, 1280)


## 1st try


In [6]:
# import tensorflow as tf
# import numpy as np

# def visualize_neuron(model, neuron_index, steps=120, lr=0.06):
#     # Feature extractor (GAP outputs)
#     feature_extractor = tf.keras.Model(
#         inputs=model.input,
#         outputs=model.get_layer('global_average_pooling2d').output
#     )

#     # Start from random noise image
#     img = tf.random.uniform((1, 224, 224, 3))

#     for _ in range(steps):
#         with tf.GradientTape() as tape:
#             tape.watch(img)
#             features = feature_extractor(img)
#             loss = features[0, neuron_index]  # maximize this neuron

#         grads = tape.gradient(loss, img)
#         grads = tf.math.l2_normalize(grads)
#         img += grads * lr

#     img = tf.clip_by_value(img[0], 0, 1)
#     return img.numpy()


# img_neuron_42 = visualize_neuron(model, neuron_index=42)

# import matplotlib.pyplot as plt
# plt.imshow(img_neuron_42)
# plt.title("Feature 42 - What does it detect?")
# plt.axis("off")
# plt.show()


In [7]:
# import numpy as np
# import shap

# def shap_explain(model, background_imgs, target_img, class_names):
    
#     explainer = shap.GradientExplainer(model, background_imgs)

#     # GradientExplainer binary output -> returns single SHAP value map
#     shap_values = explainer.shap_values(target_img)
    
#     # shap_values is a list with 1 element for binary classification
#     shap_map = shap_values[0]   # shape: (1, 224, 224, 3)
    
#     # Tahmin yapalım
#     pred = model.predict(target_img)[0][0]

#     # sınıf ismi belirle
#     predicted_class = class_names[1] if pred > 0.5 else class_names[0]

#     # Görselleştirme
#     shap.image_plot(
#         [shap_map],        # Tek bir map
#         target_img[0],     # Orijinal görüntü
#         labels=[predicted_class]
#     )


## 2nd try


In [12]:
import tensorflow as tf
import numpy as np
import shap
import matplotlib.pyplot as plt

# 1. Modeli yükle
model_path = "/Users/ozgeyavuz/490/cat_or_dog_model_dataset/cats-vs-dogs-classifier-tensorflow2-fine-tuned-mobilenetv2-v1/cats_dogs_finetuned_FT.keras"
model = tf.keras.models.load_model(model_path)

# 2. Feature extractor (1280-dim)
feature_extractor = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer('global_average_pooling2d').output
)

# 3. Sınıflandırıcıyı ayır
classifier_input = tf.keras.Input(shape=(1280,))
x = model.get_layer('dense')(classifier_input)
classifier = tf.keras.Model(classifier_input, x)


# ======================================
#   SHAP — FEATURE LEVEL
# ======================================

def shap_feature_importance(model, feature_extractor, background_imgs, target_img, class_names):

    # Background görüntüleri → feature space'e yansıt
    background_features = feature_extractor.predict(background_imgs)

    # Target image → feature vector (1×1280)
    target_features = feature_extractor.predict(target_img)

    # SHAP Kernel Explainer
    explainer = shap.KernelExplainer(classifier.predict, background_features)

    # Her sınıf için 1280 SHAP değeri döner
    shap_values = explainer(target_features)
    shap.plots.waterfall(shap_values[0][0], max_display=20)

    # Tahmin edilen sınıf
    # pred = classifier.predict(target_features)[0]
    # winner = np.argmax(pred)

    # # Winner sınıfın SHAP değerleri (1280,)
    # sv = shap_values[winner][0]

    # # En önemli 20 özelliği seç
    # top_k = 20
    # idx = np.argsort(np.abs(sv))[-top_k:][::-1]

    # plt.figure(figsize=(10, 6))
    # plt.barh([f"F{n}" for n in idx], sv[idx])
    # plt.title(f"Top {top_k} Most Important Features For Class: {class_names[winner]}")
    # plt.xlabel("SHAP Value (impact)")
    # plt.gca().invert_yaxis()
    # plt.show()

    # return idx, sv


In [13]:

# background images (5–20 images önerilir)
background = np.concatenate([load_and_preprocess("/Users/ozgeyavuz/490/cat_or_dog_model_dataset/PetImages/test/cats/cat_1.jpg"),
                             load_and_preprocess("/Users/ozgeyavuz/490/cat_or_dog_model_dataset/PetImages/test/cats/cat_5.jpg"),load_and_preprocess("/Users/ozgeyavuz/490/cat_or_dog_model_dataset/PetImages/test/cats/cat_60.jpg"),
                             load_and_preprocess("/Users/ozgeyavuz/490/cat_or_dog_model_dataset/PetImages/test/cats/cat_5.jpg"),load_and_preprocess("/Users/ozgeyavuz/490/cat_or_dog_model_dataset/PetImages/test/cats/cat_60.jpg"),
                             load_and_preprocess("/Users/ozgeyavuz/490/cat_or_dog_model_dataset/PetImages/test/cats/cat_124.jpg"),load_and_preprocess("/Users/ozgeyavuz/490/cat_or_dog_model_dataset/PetImages/test/cats/cat_234.jpg"),
                             load_and_preprocess("/Users/ozgeyavuz/490/cat_or_dog_model_dataset/PetImages/test/cats/cat_342.jpg"), load_and_preprocess("/Users/ozgeyavuz/490/cat_or_dog_model_dataset/PetImages/test/cats/cat_417.jpg"),
                             load_and_preprocess("/Users/ozgeyavuz/490/cat_or_dog_model_dataset/PetImages/test/cats/cat_496.jpg"), load_and_preprocess("/Users/ozgeyavuz/490/cat_or_dog_model_dataset/PetImages/test/cats/cat_574.jpg")])

# target image
img = load_and_preprocess("/Users/ozgeyavuz/490/cat_or_dog_model_dataset/PetImages/test/cats/cat_595.jpg")

shap_feature_importance(model, feature_extractor, background, img, ['Cat', 'Dog'])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 335ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step


  0%|          | 0/1 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1583/1583 ━━━━━━━━━━━━━━━━━━━━ 0s 202us/step


100%|██████████| 1/1 [00:00<00:00,  1.36it/s]


IndexError: invalid index to scalar variable.

<Figure size 800x200 with 0 Axes>

## 🧾 Model Evaluation Summary

The fine-tuned **MobileNetV2 (ImageNet weights)** model was tested on a **different Cats vs Dogs dataset**  
(`/kaggle/input/cats-and-dogs-image-classification/`), which includes separate `train/` and `test/` directories.

Despite being trained on a different dataset (the *Dog and Cat Classification Dataset*),  
the model maintained **strong generalization performance**:

| Metric | Cats | Dogs | Overall |
|:--|:--:|:--:|:--:|
| **Precision** | 0.98 | 0.92 | — |
| **Recall** | 0.92 | 0.98 | — |
| **F1-score** | 0.95 | 0.95 | **0.95** |
| **Accuracy** | — | — | **94.8%** |

---

### 💡 Insights
- The model achieves **~95% accuracy** on a completely unseen dataset —  
  demonstrating excellent **transferability** and **robustness** to distribution shifts.
- Slightly lower recall for *cats* indicates occasional confusion between close-up cat faces and dogs.
- The model’s performance drop (~4% from validation) is normal and expected due to dataset differences.

---

### ✅ Key Takeaways
- **Architecture:** MobileNetV2 (fine-tuned top 30 layers, dropout 0.2)  
- **Input size:** 224×224×3 RGB  
- **Inference speed:** < 60 ms per image on GPU  
- **Generalization:** Excellent — minimal overfitting, stable across datasets  

Overall, this experiment confirms that the fine-tuned MobileNetV2 is a **lightweight, accurate, and highly portable** model for binary pet image classification.
